# ABO Data Inspection Notebook

## 1. Project and Inspection Goal

This notebook supports **Enterprise AI/ML Engineering Framework v2.0**, Stage 2: Data, Step 5: Data Understanding & Schema Inspection.

The goal is to inspect the real Amazon Berkeley Objects (ABO) raw listing and image archives before deciding production cleaning rules.

Scope boundaries:

- Inspect raw ABO metadata and image archive structure only.
- Use bounded sampling suitable for a laptop.
- Do not extract full tar archives.
- Do not load the full dataset into memory.
- Do not create cleaned data tables in this task.
- Keep ABO separate from RetailRocket data.

## 2. Imports and Path Setup

In [38]:
from __future__ import annotations

import gzip
import json
import tarfile
from collections import Counter
from itertools import islice
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve().parent

ABO_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "amazon_berkeley_text_images-based"
LISTINGS_TAR_PATH = ABO_RAW_DIR / "abo-listings.tar"
IMAGES_TAR_PATH = ABO_RAW_DIR / "abo-images-small.tar"
SAMPLE_ABO_DIR = PROJECT_ROOT / "data" / "sample" / "amazon_berkeley_objects"

MAX_TAR_MEMBERS_TO_DISPLAY = 30
MAX_LISTING_RECORDS = 500

paths = {
    "project_root": PROJECT_ROOT,
    "abo_raw_dir": ABO_RAW_DIR,
    "listings_tar": LISTINGS_TAR_PATH,
    "images_tar": IMAGES_TAR_PATH,
    "sample_abo_dir": SAMPLE_ABO_DIR,
}

paths

{'project_root': PosixPath('/mnt/d/my_AI_projects/enterprise-multimodal-ecommerce-recommender'),
 'abo_raw_dir': PosixPath('/mnt/d/my_AI_projects/enterprise-multimodal-ecommerce-recommender/data/raw/amazon_berkeley_text_images-based'),
 'listings_tar': PosixPath('/mnt/d/my_AI_projects/enterprise-multimodal-ecommerce-recommender/data/raw/amazon_berkeley_text_images-based/abo-listings.tar'),
 'images_tar': PosixPath('/mnt/d/my_AI_projects/enterprise-multimodal-ecommerce-recommender/data/raw/amazon_berkeley_text_images-based/abo-images-small.tar'),
 'sample_abo_dir': PosixPath('/mnt/d/my_AI_projects/enterprise-multimodal-ecommerce-recommender/data/sample/amazon_berkeley_objects')}

## 3. Confirm Raw Data Files Exist

In [39]:
def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def describe_path(path: Path) -> dict[str, Any]:
    """Return a small, safe path summary without reading file contents."""
    exists = path.exists()
    return {
        "path": display_path(path),
        "exists": exists,
        "is_file": path.is_file() if exists else False,
        "is_dir": path.is_dir() if exists else False,
        "size_mb": round(path.stat().st_size / (1024 * 1024), 2) if exists and path.is_file() else None,
    }


pd.DataFrame([describe_path(path) for path in paths.values()])

,path,exists,is_file,is_dir,size_mb
0,.,True,False,True,NaN
1,data/raw/amazon_berkeley_text_images-based,True,False,True,NaN
2,data/raw/amazon_berkeley_text_images-based/abo-listings.tar,True,True,False,83.43
3,data/raw/amazon_berkeley_text_images-based/abo-images-small.tar,True,True,False,3102.67
4,data/sample/amazon_berkeley_objects,True,False,True,NaN


In [40]:
sample_files = sorted(SAMPLE_ABO_DIR.glob("**/*")) if SAMPLE_ABO_DIR.exists() else []
pd.DataFrame(
    [describe_path(path) for path in sample_files[:MAX_TAR_MEMBERS_TO_DISPLAY]]
) if sample_files else pd.DataFrame(columns=["path", "exists", "is_file", "is_dir", "size_mb"])

,path,exists,is_file,is_dir,size_mb
0,data/sample/amazon_berkeley_objects/image_paths_sample.txt,True,True,False,0.0
1,data/sample/amazon_berkeley_objects/images,True,False,True,NaN
2,data/sample/amazon_berkeley_objects/images/small,True,False,True,NaN
3,data/sample/amazon_berkeley_objects/images/small/a1,True,False,True,NaN
4,data/sample/amazon_berkeley_objects/images/small/a1/a1b2c3d4.jpg,True,True,False,0.0
5,data/sample/amazon_berkeley_objects/images/small/a1/a1b2c3d5.jpg,True,True,False,0.0
6,data/sample/amazon_berkeley_objects/images/small/b2,True,False,True,NaN
7,data/sample/amazon_berkeley_objects/images/small/b2/b2c3d4e5.jpg,True,True,False,0.0
8,data/sample/amazon_berkeley_objects/images/small/b2/b2c3d4e6.jpg,True,True,False,0.0
9,data/sample/amazon_berkeley_objects/images/small/c3,True,False,True,NaN


## 4. Inspect `abo-listings.tar` Structure

In [41]:
def inspect_tar_members(tar_path: Path, limit: int = MAX_TAR_MEMBERS_TO_DISPLAY) -> pd.DataFrame:
    """Inspect a bounded number of tar members without extracting files."""
    if not tar_path.exists():
        return pd.DataFrame(columns=["name", "size_mb", "is_file", "suffixes"])

    rows: list[dict[str, Any]] = []
    with tarfile.open(tar_path, "r") as tar:
        for member in islice(tar, limit):
            member_path = Path(member.name)
            rows.append(
                {
                    "name": member.name,
                    "size_mb": round(member.size / (1024 * 1024), 3),
                    "is_file": member.isfile(),
                    "suffixes": "".join(member_path.suffixes),
                }
            )
    return pd.DataFrame(rows)


listings_members_df = inspect_tar_members(LISTINGS_TAR_PATH)
listings_members_df

,name,size_mb,is_file,suffixes
0,LICENSE-CC-BY-4.0.txt,0.014,True,.0.txt
1,listings,0.000,False,
2,listings/README.md,0.007,True,.md
3,listings/metadata,0.000,False,
4,listings/metadata/listings_7.json.gz,5.177,True,.json.gz
5,listings/metadata/listings_4.json.gz,5.289,True,.json.gz
6,listings/metadata/listings_2.json.gz,5.194,True,.json.gz
7,listings/metadata/listings_c.json.gz,5.191,True,.json.gz
8,listings/metadata/listings_6.json.gz,5.242,True,.json.gz
9,listings/metadata/listings_0.json.gz,5.194,True,.json.gz


In [42]:
def count_tar_members_by_suffix(tar_path: Path, max_members: int | None = None) -> pd.DataFrame:
    """Count member suffixes with optional bounded traversal."""
    if not tar_path.exists():
        return pd.DataFrame(columns=["suffixes", "member_count"])

    counter: Counter[str] = Counter()
    with tarfile.open(tar_path, "r") as tar:
        iterator = tar if max_members is None else islice(tar, max_members)
        for member in iterator:
            if member.isfile():
                counter["".join(Path(member.name).suffixes) or "<no_suffix>"] += 1

    return pd.DataFrame(counter.items(), columns=["suffixes", "member_count"]).sort_values(
        "member_count", ascending=False
    )


count_tar_members_by_suffix(LISTINGS_TAR_PATH)

,suffixes,member_count
2,.json.gz,16
0,.0.txt,1
1,.md,1


## 5. Inspect `abo-images-small.tar` Structure

In [43]:
images_members_df = inspect_tar_members(IMAGES_TAR_PATH)
images_members_df

,name,size_mb,is_file,suffixes
0,LICENSE-CC-BY-4.0.txt,0.014,True,.0.txt
1,images,0.000,False,
2,images/small,0.000,False,
3,images/small/00,0.000,False,
4,images/small/00/00834536.jpg,0.009,True,.jpg
5,images/small/00/001042f3.jpg,0.007,True,.jpg
6,images/small/00/007fb0e2.jpg,0.013,True,.jpg
7,images/small/00/00a627c8.jpg,0.012,True,.jpg
8,images/small/00/004f0a7d.jpg,0.005,True,.jpg
9,images/small/00/00c69266.jpg,0.003,True,.jpg


In [44]:
image_suffix_counts_df = count_tar_members_by_suffix(IMAGES_TAR_PATH)
image_suffix_counts_df

,suffixes,member_count
1,.jpg,398210
2,.png,2
0,.0.txt,1
3,.md,1
4,.csv.gz,1


In [45]:
def collect_image_ids_from_tar(tar_path: Path, max_members: int | None = None) -> set[str]:
    """Collect image ID stems from image tar member names without extracting images."""
    image_ids: set[str] = set()
    if not tar_path.exists():
        return image_ids

    with tarfile.open(tar_path, "r") as tar:
        iterator = tar if max_members is None else islice(tar, max_members)
        for member in iterator:
            if member.isfile():
                image_ids.add(Path(member.name).stem)
    return image_ids


available_image_ids = collect_image_ids_from_tar(IMAGES_TAR_PATH)
len(available_image_ids)

398215

## 6. Read a Small Sample of Listing Records

In [46]:
def read_json_records_from_member(file_obj: Any, member_name: str, max_records: int) -> list[dict[str, Any]]:
    """Read bounded JSON records from a tar member that may be JSON, JSONL, or gzipped JSONL."""
    stream = gzip.GzipFile(fileobj=file_obj) if member_name.endswith(".gz") else file_obj
    raw = stream.read(1024 * 1024 * 8)
    text = raw.decode("utf-8", errors="replace").strip()
    if not text:
        return []

    records: list[dict[str, Any]] = []

    # Many metadata archives use JSON Lines; try that before whole-file JSON.
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            parsed = json.loads(line)
        except json.JSONDecodeError:
            records = []
            break
        if isinstance(parsed, dict):
            records.append(parsed)
        if len(records) >= max_records:
            return records

    if records:
        return records[:max_records]

    parsed = json.loads(text)
    if isinstance(parsed, list):
        return [record for record in parsed[:max_records] if isinstance(record, dict)]
    if isinstance(parsed, dict):
        for value in parsed.values():
            if isinstance(value, list):
                return [record for record in value[:max_records] if isinstance(record, dict)]
        return [parsed]
    return []


def sample_listing_records(tar_path: Path, max_records: int = MAX_LISTING_RECORDS) -> list[dict[str, Any]]:
    """Sample listing records from the first readable metadata member in the listings archive."""
    if not tar_path.exists():
        return []

    records: list[dict[str, Any]] = []
    with tarfile.open(tar_path, "r") as tar:
        for member in tar:
            if not member.isfile():
                continue
            if not any(member.name.endswith(suffix) for suffix in (".json", ".jsonl", ".json.gz", ".jsonl.gz")):
                continue
            file_obj = tar.extractfile(member)
            if file_obj is None:
                continue
            records = read_json_records_from_member(file_obj, member.name, max_records)
            if records:
                print(f"Sampled {len(records)} records from {member.name}")
                return records
    return records


listing_records = sample_listing_records(LISTINGS_TAR_PATH)
listings_df = pd.DataFrame(listing_records)
listings_df.head(3)

Sampled 500 records from listings/metadata/listings_7.json.gz


,brand,bullet_point,color,fabric_type,item_id,item_name,item_weight,model_name,model_number,product_type,style,main_image_id,other_image_id,color_code,country,marketplace,domain_name,material,item_keywords,node,item_dimensions,pattern,model_year,product_description,spin_id,3dmodel_id,item_shape,finish_type
0,"[{'language_tag': 'fr_FR', 'value': 'Amazon Essentials'}]","[{'language_tag': 'fr_FR', 'value': 'Plat classique et polyvalent conçu pour un usage quotidien et un ajustement sup...","[{'language_tag': 'fr_FR', 'standardized_values': ['Marron'], 'value': 'Peau'}]","[{'language_tag': 'fr_FR', 'value': '100% Synthétique'}]",B07NQ437BB,"[{'language_tag': 'fr_FR', 'value': 'Amazon Essentials Manny Footwear, Peau, 11 M US'}]","[{'normalized_value': {'unit': 'pounds', 'value': 0.26875}, 'unit': 'ounces', 'value': 4.3}]","[{'language_tag': 'fr_FR', 'value': 'Manny'}]",[{'value': 'Manny'}],[{'value': 'SHOES'}],"[{'language_tag': 'fr_FR', 'value': 'Manny'}]",61fH9aTfMIL,"[71AymDrpuFL, 61qXoMZjStL, 61qDo19NgzL, 61h3BqlKrNL, 61QPaJdIxaL, 71g+H66qBML, 61-Zjl2vLzL]",[#A87C6D],FR,Amazon,amazon.fr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo'}]","[{'language_tag': 'en_IN', 'value': 'Snug fit for Samsung Galaxy J2 Ace, with perfect cut-outs for volume buttons, a...","[{'language_tag': 'en_IN', 'standardized_values': ['multi-colored'], 'value': 'Multicolor'}]",NaN,B0857LSVB7,"[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo Designer Lion UV Printed Soft Back Case Mobile Cover for ...","[{'normalized_value': {'unit': 'pounds', 'value': 0.110231131}, 'unit': 'grams', 'value': 50}]","[{'language_tag': 'en_IN', 'value': 'Samsung Galaxy J2 Ace'}]",[{'value': 'UV10392-SL40350'}],[{'value': 'CELLULAR_PHONE_CASE'}],NaN,81-DuD5XzmL,"[61+woWTqkwL, 61SE4RTPjdL]",NaN,IN,Amazon,amazon.in,"[{'language_tag': 'en_IN', 'value': 'Silicon'}]","[{'language_tag': 'en_IN', 'value': 'Back Cover'}, {'language_tag': 'en_IN', 'value': 'Designer Case'}, {'language_t...","[{'node_id': 12538061031, 'node_name': '/Categories/Mobiles & Accessories/Mobile Accessories/Maintenance, Upkeep & R...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[{'language_tag': 'en_SG', 'value': 'AmazonBasics'}]","[{'language_tag': 'en_SG', 'value': 'For best performance, follow the manufacturer's recommendations in your vehicle...",NaN,NaN,B07C5FF8QS,"[{'language_tag': 'en_SG', 'value': 'AmazonBasics High Mileage Motor Oil, Synthetic Blend, 10W-30, 5 Quart'}]",NaN,NaN,[{'value': 'AM13BH3Q'}],[{'value': 'AUTO_OIL'}],"[{'language_tag': 'en_SG', 'value': 'High Mileage - Synthetic Blend'}]",81YCp3dcurL,"[817GQ6xx-QL, 81Vr9poKgCL]",NaN,SG,Amazon,amazon.sg,NaN,"[{'language_tag': 'en_SG', 'value': 'oil'}, {'language_tag': 'en_SG', 'value': 'mobil 1'}, {'language_tag': 'en_SG',...","[{'node_id': 6394587051, 'node_name': '/Categories/Oils & Fluids/Additives'}]","{'height': {'normalized_value': {'unit': 'inches', 'value': 12.5}, 'unit': 'inches', 'value': 12.5}, 'length': {'nor...","[{'language_tag': 'en_SG', 'value': '10W-30'}]",NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
listings_df.shape

(500, 28)

## 7. Display Available Fields / Nested Fields

In [48]:
def summarize_fields(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    field_counter: Counter[str] = Counter()
    type_counter: dict[str, Counter[str]] = {}

    for record in records:
        for key, value in record.items():
            field_counter[key] += 1
            type_counter.setdefault(key, Counter())[type(value).__name__] += 1

    for key, count in field_counter.items():
        rows.append(
            {
                "field": key,
                "present_count": count,
                "present_rate": round(count / max(len(records), 1), 3),
                "observed_types": dict(type_counter[key]),
            }
        )
    return pd.DataFrame(rows).sort_values("field") if rows else pd.DataFrame()


fields_df = summarize_fields(listing_records)
fields_df

,field,present_count,present_rate,observed_types
25,3dmodel_id,33,0.066,{'str': 33}
0,brand,500,1.000,{'list': 500}
1,bullet_point,444,0.888,{'list': 444}
2,color,395,0.790,{'list': 395}
13,color_code,70,0.140,{'list': 70}
14,country,500,1.000,{'str': 500}
16,domain_name,500,1.000,{'str': 500}
3,fabric_type,33,0.066,{'list': 33}
27,finish_type,4,0.008,{'list': 4}
20,item_dimensions,144,0.288,{'dict': 144}


In [49]:
def flatten_keys(value: Any, prefix: str = "") -> set[str]:
    keys: set[str] = set()
    if isinstance(value, dict):
        for key, nested_value in value.items():
            full_key = f"{prefix}.{key}" if prefix else str(key)
            keys.add(full_key)
            keys.update(flatten_keys(nested_value, full_key))
    elif isinstance(value, list):
        for item in value[:3]:
            keys.update(flatten_keys(item, prefix))
    return keys


nested_keys = sorted({key for record in listing_records[:50] for key in flatten_keys(record)})
pd.DataFrame({"nested_field": nested_keys})

,nested_field
0,3dmodel_id
1,brand
2,brand.language_tag
3,brand.value
4,bullet_point
...,...
75,product_type.value
76,spin_id
77,style
78,style.language_tag


## 8. Inspect Product ID Fields

In [50]:
def first_scalar(value: Any) -> Any:
    if isinstance(value, list):
        return value[0] if value else None
    return value


id_candidate_fields = [field for field in listings_df.columns if "id" in field.lower()]
id_summary_rows = []
for field in id_candidate_fields:
    series = listings_df[field].map(first_scalar)
    id_summary_rows.append(
        {
            "field": field,
            "non_null_count": int(series.notna().sum()),
            "unique_count": int(series.dropna().nunique()),
            "example_values": series.dropna().astype(str).head(5).tolist(),
        }
    )

pd.DataFrame(id_summary_rows)

,field,non_null_count,unique_count,example_values
0,item_id,500,500,"[B07NQ437BB, B0857LSVB7, B07C5FF8QS, B07K591232, B07TG425LX]"
1,main_image_id,495,491,"[61fH9aTfMIL, 81-DuD5XzmL, 81YCp3dcurL, 71EwvyC4A1L, 71ZoXfUr-sL]"
2,other_image_id,462,292,"[71AymDrpuFL, 61+woWTqkwL, 817GQ6xx-QL, 71FbjGLOOIL, 81IiaAkE6UL]"
3,spin_id,30,30,"[fd0df7b1, a561bd11, 71235c09, d3fb49a0, a6deb0c9]"
4,3dmodel_id,33,33,"[B07DBJQC1K, B07M7NSYD8, B075HX5QD6, B086B5P473, B07QJMFCSJ]"


## 9. Inspect Text Fields

Target fields: `item_name`, `brand`, `bullet_point`, `product_type`, `color`, `material`, and `style`.

In [51]:
TEXT_FIELDS = ["item_name", "brand", "bullet_point", "product_type", "color", "material", "style"]


def normalize_text_value(value: Any) -> str:
    if value is None:
        return ""
    if not isinstance(value, (list, dict)) and pd.isna(value):
        return ""
    if isinstance(value, list):
        return " ".join(normalize_text_value(item) for item in value).strip()
    if isinstance(value, dict):
        return " ".join(normalize_text_value(item) for item in value.values()).strip()
    return str(value).strip()


def combined_text(record: dict[str, Any], fields: list[str] = TEXT_FIELDS) -> str:
    return " ".join(normalize_text_value(record.get(field)) for field in fields).strip()


text_summary_rows = []
for field in TEXT_FIELDS:
    if field not in listings_df.columns:
        text_summary_rows.append({"field": field, "exists": False})
        continue
    normalized = listings_df[field].map(normalize_text_value)
    text_summary_rows.append(
        {
            "field": field,
            "exists": True,
            "non_empty_count": int((normalized.str.len() > 0).sum()),
            "non_empty_rate": round(float((normalized.str.len() > 0).mean()), 3),
            "example_values": normalized[normalized.str.len() > 0].head(3).tolist(),
        }
    )

pd.DataFrame(text_summary_rows)

,field,exists,non_empty_count,non_empty_rate,example_values
0,item_name,True,500,1.000,"[fr_FR Amazon Essentials Manny Footwear, Peau, 11 M US, en_IN Amazon Brand - Solimo Designer Lion UV Printed Soft Ba..."
1,brand,True,500,1.000,"[fr_FR Amazon Essentials, en_IN Amazon Brand - Solimo, en_SG AmazonBasics]"
2,bullet_point,True,444,0.888,[fr_FR Plat classique et polyvalent conçu pour un usage quotidien et un ajustement supérieur. fr_FR Silhouette raffi...
3,product_type,True,500,1.000,"[SHOES, CELLULAR_PHONE_CASE, AUTO_OIL]"
4,color,True,395,0.790,"[fr_FR Marron Peau, en_IN multi-colored Multicolor, en_IN multi-colored Others]"
5,material,True,188,0.376,"[en_IN Silicon, ml_IN കോട്ടൺ en_IN cotton hi_IN कॉटन, en_IN Silicone]"
6,style,True,142,0.284,"[fr_FR Manny, en_SG High Mileage - Synthetic Blend, ml_IN സ്ട്രേറ്റ് en_IN straight hi_IN स्ट्रेट]"


In [52]:
text_profile_df = pd.DataFrame(
    {
        "item_id": listings_df["item_id"].map(first_scalar) if "item_id" in listings_df.columns else None,
        "combined_text": [combined_text(record) for record in listing_records],
    }
)
text_profile_df["combined_text_length"] = text_profile_df["combined_text"].str.len()
text_profile_df.describe(include="all")

,item_id,combined_text,combined_text_length
count,500,500,500.000000
unique,500,499,NaN
top,B07NQ437BB,en_IN Amazon Brand - Solimo Designer Abstract 3D Printed Hard Back Case Mobile Cover for Realme 3 / Realme 3i en_IN ...,NaN
freq,1,2,NaN
mean,NaN,NaN,754.324000
std,NaN,NaN,785.381719
min,NaN,NaN,66.000000
25%,NaN,NaN,494.500000
50%,NaN,NaN,632.000000
75%,NaN,NaN,695.250000


## 10. Inspect Image Fields

Target fields: `main_image_id` and other image-related fields if available.

In [53]:
image_candidate_fields = [field for field in listings_df.columns if "image" in field.lower()]
image_summary_rows = []
for field in image_candidate_fields:
    normalized = listings_df[field].map(normalize_text_value)
    image_summary_rows.append(
        {
            "field": field,
            "non_empty_count": int((normalized.str.len() > 0).sum()),
            "non_empty_rate": round(float((normalized.str.len() > 0).mean()), 3),
            "example_values": normalized[normalized.str.len() > 0].head(5).tolist(),
        }
    )

pd.DataFrame(image_summary_rows)

,field,non_empty_count,non_empty_rate,example_values
0,main_image_id,495,0.990,"[61fH9aTfMIL, 81-DuD5XzmL, 81YCp3dcurL, 71EwvyC4A1L, 71ZoXfUr-sL]"
1,other_image_id,462,0.924,"[71AymDrpuFL 61qXoMZjStL 61qDo19NgzL 61h3BqlKrNL 61QPaJdIxaL 71g+H66qBML 61-Zjl2vLzL, 61+woWTqkwL 61SE4RTPjdL, 817GQ..."


## 11. Check Missing Values in Important Fields

In [54]:
IMPORTANT_FIELDS = ["item_id", *TEXT_FIELDS, "main_image_id", "other_image_id"]

missing_rows = []
for field in IMPORTANT_FIELDS:
    if field not in listings_df.columns:
        missing_rows.append({"field": field, "exists": False, "missing_count": None, "missing_rate": None})
        continue
    normalized = listings_df[field].map(normalize_text_value)
    missing = normalized.str.len() == 0
    missing_rows.append(
        {
            "field": field,
            "exists": True,
            "missing_count": int(missing.sum()),
            "missing_rate": round(float(missing.mean()), 3),
        }
    )

pd.DataFrame(missing_rows)

,field,exists,missing_count,missing_rate
0,item_id,True,0,0.000
1,item_name,True,0,0.000
2,brand,True,0,0.000
3,bullet_point,True,56,0.112
4,product_type,True,0,0.000
5,color,True,105,0.210
6,material,True,312,0.624
7,style,True,358,0.716
8,main_image_id,True,5,0.010
9,other_image_id,True,38,0.076


## 12. Check Duplicate Product IDs

In [55]:
if "item_id" in listings_df.columns:
    item_ids = listings_df["item_id"].map(first_scalar).dropna().astype(str)
    duplicate_item_ids = item_ids[item_ids.duplicated(keep=False)]
    duplicate_summary = {
        "sample_records": len(listings_df),
        "non_null_item_ids": int(item_ids.shape[0]),
        "unique_item_ids": int(item_ids.nunique()),
        "duplicate_item_id_rows": int(duplicate_item_ids.shape[0]),
    }
else:
    duplicate_summary = {
        "sample_records": len(listings_df),
        "non_null_item_ids": 0,
        "unique_item_ids": 0,
        "duplicate_item_id_rows": None,
    }

duplicate_summary

{'sample_records': 500,
 'non_null_item_ids': 500,
 'unique_item_ids': 500,
 'duplicate_item_id_rows': 0}

## 13. Check Image Mapping Feasibility

In [56]:
def image_ids_from_record(record: dict[str, Any]) -> set[str]:
    image_ids: set[str] = set()
    for field in image_candidate_fields:
        value = record.get(field)
        if isinstance(value, list):
            image_ids.update(str(item).strip() for item in value if str(item).strip())
        elif value is not None:
            text = str(value).strip()
            if text:
                image_ids.add(text)
    return image_ids


image_mapping_rows = []
for record in listing_records:
    record_image_ids = image_ids_from_record(record)
    image_mapping_rows.append(
        {
            "item_id": first_scalar(record.get("item_id")),
            "image_id_count": len(record_image_ids),
            "has_main_image_id": bool(normalize_text_value(record.get("main_image_id"))),
            "mapped_image_count": len(record_image_ids & available_image_ids),
        }
    )

image_mapping_df = pd.DataFrame(image_mapping_rows)
image_mapping_df.describe() if not image_mapping_df.empty else image_mapping_df

,image_id_count,mapped_image_count
count,500.000000,500.0
mean,4.826000,0.0
std,2.065977,0.0
min,0.000000,0.0
25%,3.750000,0.0
50%,5.000000,0.0
75%,6.000000,0.0
max,14.000000,0.0


In [57]:
image_mapping_df.head(10)

,item_id,image_id_count,has_main_image_id,mapped_image_count
0,B07NQ437BB,8,True,0
1,B0857LSVB7,3,True,0
2,B07C5FF8QS,3,True,0
3,B07K591232,4,True,0
4,B07TG425LX,5,True,0
5,B07LCHFZCW,3,True,0
6,B077W2YX72,5,True,0
7,B07TH39LDF,5,True,0
8,B07TG3WCBD,5,True,0
9,B07TL47SLR,4,False,0


## 13A. Inspect Image CSV Mapping Metadata

The direct image mapping check above compares listing image IDs to image file stems in `abo-images-small.tar`. Because the sampled `mapped_image_count` is zero, inspect any `.csv.gz` metadata members inside the image archive to see whether they provide the bridge from listing image IDs to real image paths.

In [58]:
def list_csv_gz_members(tar_path: Path) -> list[str]:
    """List compressed CSV members inside a tar archive without extracting them."""
    if not tar_path.exists():
        return []

    csv_members: list[str] = []
    with tarfile.open(tar_path, "r") as tar:
        for member in tar:
            if member.isfile() and member.name.endswith(".csv.gz"):
                csv_members.append(member.name)
    return csv_members


image_csv_gz_members = list_csv_gz_members(IMAGES_TAR_PATH)
pd.DataFrame({"csv_gz_member": image_csv_gz_members})

,csv_gz_member
0,images/metadata/images.csv.gz


In [59]:
def read_csv_gz_member_sample(tar_path: Path, member_name: str, nrows: int = 10) -> pd.DataFrame:
    """Read a small sample from a .csv.gz member inside a tar archive."""
    if not tar_path.exists():
        return pd.DataFrame()

    with tarfile.open(tar_path, "r") as tar:
        member = tar.getmember(member_name)
        file_obj = tar.extractfile(member)
        if file_obj is None:
            return pd.DataFrame()
        with gzip.GzipFile(fileobj=file_obj) as gzip_file:
            return pd.read_csv(gzip_file, nrows=nrows)


image_csv_samples: dict[str, pd.DataFrame] = {}
for member_name in image_csv_gz_members:
    sample_df = read_csv_gz_member_sample(IMAGES_TAR_PATH, member_name, nrows=10)
    image_csv_samples[member_name] = sample_df
    print(f"CSV member: {member_name}")
    print(f"Sample shape: {sample_df.shape}")
    print(f"Columns: {sample_df.columns.tolist()}")
    display(sample_df.head())

CSV member: images/metadata/images.csv.gz
Sample shape: (10, 4)
Columns: ['image_id', 'height', 'width', 'path']


,image_id,height,width,path
0,010-mllS7JL,106,106,14/14fe8812.jpg
1,01dkn0Gyx0L,122,122,da/daab0cad.jpg
2,01sUPg0387L,111,111,d2/d2daaae9.jpg
3,1168jc-5r1L,186,186,3a/3a4e88e6.jpg
4,11RUV5Fs65L,30,500,d9/d91ab9cf.jpg


In [60]:
def classify_image_csv_columns(columns: list[str]) -> dict[str, list[str]]:
    """Identify likely image ID and file path columns from CSV column names."""
    id_terms = ("image_id", "imageid", "image", "id")
    path_terms = ("path", "file", "filename", "url", "location")

    id_columns = [column for column in columns if any(term in column.lower() for term in id_terms)]
    path_columns = [column for column in columns if any(term in column.lower() for term in path_terms)]
    return {"possible_image_id_columns": id_columns, "possible_path_columns": path_columns}


csv_column_summary_rows = []
for member_name, sample_df in image_csv_samples.items():
    classified_columns = classify_image_csv_columns(sample_df.columns.tolist())
    csv_column_summary_rows.append(
        {
            "csv_filename": member_name,
            "sample_rows": int(sample_df.shape[0]),
            "sample_columns": int(sample_df.shape[1]),
            "possible_image_id_columns": classified_columns["possible_image_id_columns"],
            "possible_path_columns": classified_columns["possible_path_columns"],
        }
    )

image_csv_column_summary_df = pd.DataFrame(csv_column_summary_rows)
image_csv_column_summary_df

,csv_filename,sample_rows,sample_columns,possible_image_id_columns,possible_path_columns
0,images/metadata/images.csv.gz,10,4,"[image_id, width]",[path]


In [61]:
def sample_main_image_ids(records: list[dict[str, Any]], limit: int = 25) -> set[str]:
    """Collect a small set of listing main_image_id values from sampled listing records."""
    values: set[str] = set()
    for record in records:
        value = record.get("main_image_id")
        if isinstance(value, list):
            values.update(str(item).strip() for item in value if str(item).strip())
        elif value is not None:
            text = str(value).strip()
            if text:
                values.add(text)
        if len(values) >= limit:
            break
    return set(sorted(values)[:limit])


listing_main_image_id_sample = sample_main_image_ids(listing_records)
pd.DataFrame({"sample_main_image_id": sorted(listing_main_image_id_sample)})

,sample_main_image_id
0,41fqYpFf6sL
1,517Y+AGBTyL
2,61M73+H74pL
3,61bCuBtuZ4L
4,61fH9aTfMIL
5,61gjqv+72vL
6,61oZajNgA-L
7,61swawiifYL
8,710GWe3VMGL
9,710vi0E1ccL


In [62]:
MAX_IMAGE_CSV_ROWS_TO_SCAN = 100_000
IMAGE_CSV_CHUNKSIZE = 10_000


def scan_csv_member_for_image_id_matches(
    tar_path: Path,
    member_name: str,
    target_ids: set[str],
    max_rows: int = MAX_IMAGE_CSV_ROWS_TO_SCAN,
    chunksize: int = IMAGE_CSV_CHUNKSIZE,
) -> pd.DataFrame:
    """Scan a bounded number of rows for listing image IDs in candidate CSV columns."""
    if not target_ids or not tar_path.exists():
        return pd.DataFrame()

    matches: list[dict[str, Any]] = []
    rows_scanned = 0

    with tarfile.open(tar_path, "r") as tar:
        member = tar.getmember(member_name)
        file_obj = tar.extractfile(member)
        if file_obj is None:
            return pd.DataFrame()

        with gzip.GzipFile(fileobj=file_obj) as gzip_file:
            for chunk in pd.read_csv(gzip_file, chunksize=chunksize):
                rows_scanned += len(chunk)
                classified_columns = classify_image_csv_columns(chunk.columns.tolist())
                candidate_columns = sorted(
                    set(classified_columns["possible_image_id_columns"] + classified_columns["possible_path_columns"])
                )
                for column in candidate_columns:
                    values = chunk[column].dropna().astype(str)
                    matched_values = sorted(set(values) & target_ids)
                    if matched_values:
                        examples = chunk[chunk[column].astype(str).isin(matched_values)].head(5).to_dict("records")
                        matches.append(
                            {
                                "csv_filename": member_name,
                                "matched_column": column,
                                "matched_main_image_ids": matched_values,
                                "example_rows": examples,
                                "rows_scanned": min(rows_scanned, max_rows),
                            }
                        )
                if rows_scanned >= max_rows:
                    break

    return pd.DataFrame(matches)


image_csv_match_frames = [
    scan_csv_member_for_image_id_matches(IMAGES_TAR_PATH, member_name, listing_main_image_id_sample)
    for member_name in image_csv_gz_members
]
image_csv_match_df = pd.concat(image_csv_match_frames, ignore_index=True) if image_csv_match_frames else pd.DataFrame()
image_csv_match_df

,csv_filename,matched_column,matched_main_image_ids,example_rows,rows_scanned
0,images/metadata/images.csv.gz,image_id,"[41fqYpFf6sL, 517Y+AGBTyL]","[{'image_id': '41fqYpFf6sL', 'height': 476, 'width': 500, 'path': 'd6/d62740b8.jpg'}, {'image_id': '517Y+AGBTyL', 'h...",20000
1,images/metadata/images.csv.gz,image_id,[61M73+H74pL],"[{'image_id': '61M73+H74pL', 'height': 1102, 'width': 2188, 'path': '4f/4ffd9e34.jpg'}]",70000
2,images/metadata/images.csv.gz,image_id,"[61bCuBtuZ4L, 61fH9aTfMIL, 61gjqv+72vL]","[{'image_id': '61bCuBtuZ4L', 'height': 1000, 'width': 974, 'path': '87/87b38aee.jpg'}, {'image_id': '61fH9aTfMIL', '...",90000
3,images/metadata/images.csv.gz,image_id,[61oZajNgA-L],"[{'image_id': '61oZajNgA-L', 'height': 1000, 'width': 540, 'path': '55/5549b279.jpg'}]",100000


In [63]:
image_csv_mapping_summary_rows = []
for member_name, sample_df in image_csv_samples.items():
    classified_columns = classify_image_csv_columns(sample_df.columns.tolist())
    member_matches = image_csv_match_df[image_csv_match_df["csv_filename"] == member_name] if not image_csv_match_df.empty else pd.DataFrame()
    useful_columns = sorted(
        set(classified_columns["possible_image_id_columns"] + classified_columns["possible_path_columns"])
    )
    has_path_columns = bool(classified_columns["possible_path_columns"])
    image_csv_mapping_summary_rows.append(
        {
            "csv_filename": member_name,
            "useful_columns_found": useful_columns,
            "main_image_id_maps_to_csv_field": not member_matches.empty,
            "matched_columns": sorted(member_matches["matched_column"].unique().tolist()) if not member_matches.empty else [],
            "has_image_path_or_file_columns": has_path_columns,
            "recommended_next_step": (
                "Use this CSV as the candidate mapping source in the later cleaning design."
                if not member_matches.empty and has_path_columns
                else "Inspect the full CSV schema and ABO documentation before designing image mapping rules."
            ),
        }
    )

image_csv_mapping_summary_df = pd.DataFrame(image_csv_mapping_summary_rows)
image_csv_mapping_summary_df

,csv_filename,useful_columns_found,main_image_id_maps_to_csv_field,matched_columns,has_image_path_or_file_columns,recommended_next_step
0,images/metadata/images.csv.gz,"[image_id, path, width]",True,[image_id],True,Use this CSV as the candidate mapping source in the later cleaning design.


## 14. Estimate CLIP-Ready Product Count

A sampled product is considered CLIP-ready when it has usable product text and at least one image ID that maps to a file in `abo-images-small.tar`.

In [64]:
clip_ready_df = text_profile_df.join(image_mapping_df[["image_id_count", "mapped_image_count"]])
clip_ready_df["has_usable_text"] = clip_ready_df["combined_text_length"] > 0
clip_ready_df["has_mapped_image"] = clip_ready_df["mapped_image_count"] > 0
clip_ready_df["is_clip_ready"] = clip_ready_df["has_usable_text"] & clip_ready_df["has_mapped_image"]

{
    "sample_records": int(len(clip_ready_df)),
    "usable_text_count": int(clip_ready_df["has_usable_text"].sum()),
    "mapped_image_count": int(clip_ready_df["has_mapped_image"].sum()),
    "clip_ready_count": int(clip_ready_df["is_clip_ready"].sum()),
    "clip_ready_rate": round(float(clip_ready_df["is_clip_ready"].mean()), 3) if len(clip_ready_df) else 0.0,
}

{'sample_records': 500,
 'usable_text_count': 500,
 'mapped_image_count': 0,
 'clip_ready_count': 0,
 'clip_ready_rate': 0.0}

## 15. Show Good Example Records

In [65]:
display_fields = [field for field in ["item_id", *TEXT_FIELDS, "main_image_id", "other_image_id"] if field in listings_df.columns]
good_examples = listings_df.loc[clip_ready_df["is_clip_ready"], display_fields].head(5)
good_examples

,item_id,item_name,brand,bullet_point,product_type,color,material,style,main_image_id,other_image_id


## 16. Show Problematic Example Records

In [66]:
problematic_examples = listings_df.loc[~clip_ready_df["is_clip_ready"], display_fields].head(10)
problematic_examples

,item_id,item_name,brand,bullet_point,product_type,color,material,style,main_image_id,other_image_id
0,B07NQ437BB,"[{'language_tag': 'fr_FR', 'value': 'Amazon Essentials Manny Footwear, Peau, 11 M US'}]","[{'language_tag': 'fr_FR', 'value': 'Amazon Essentials'}]","[{'language_tag': 'fr_FR', 'value': 'Plat classique et polyvalent conçu pour un usage quotidien et un ajustement sup...",[{'value': 'SHOES'}],"[{'language_tag': 'fr_FR', 'standardized_values': ['Marron'], 'value': 'Peau'}]",NaN,"[{'language_tag': 'fr_FR', 'value': 'Manny'}]",61fH9aTfMIL,"[71AymDrpuFL, 61qXoMZjStL, 61qDo19NgzL, 61h3BqlKrNL, 61QPaJdIxaL, 71g+H66qBML, 61-Zjl2vLzL]"
1,B0857LSVB7,"[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo Designer Lion UV Printed Soft Back Case Mobile Cover for ...","[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo'}]","[{'language_tag': 'en_IN', 'value': 'Snug fit for Samsung Galaxy J2 Ace, with perfect cut-outs for volume buttons, a...",[{'value': 'CELLULAR_PHONE_CASE'}],"[{'language_tag': 'en_IN', 'standardized_values': ['multi-colored'], 'value': 'Multicolor'}]","[{'language_tag': 'en_IN', 'value': 'Silicon'}]",NaN,81-DuD5XzmL,"[61+woWTqkwL, 61SE4RTPjdL]"
2,B07C5FF8QS,"[{'language_tag': 'en_SG', 'value': 'AmazonBasics High Mileage Motor Oil, Synthetic Blend, 10W-30, 5 Quart'}]","[{'language_tag': 'en_SG', 'value': 'AmazonBasics'}]","[{'language_tag': 'en_SG', 'value': 'For best performance, follow the manufacturer's recommendations in your vehicle...",[{'value': 'AUTO_OIL'}],NaN,NaN,"[{'language_tag': 'en_SG', 'value': 'High Mileage - Synthetic Blend'}]",81YCp3dcurL,"[817GQ6xx-QL, 81Vr9poKgCL]"
3,B07K591232,"[{'language_tag': 'it_IT', 'value': 'AmazonBasics piumino alternativo'}]","[{'language_tag': 'it_IT', 'value': 'AmazonBasics'}]",NaN,[{'value': 'HOME_BED_AND_BATH'}],NaN,NaN,NaN,71EwvyC4A1L,"[71FbjGLOOIL, 61Ot9qEVqOL, 91WAK5rtolL]"
4,B07TG425LX,"[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo Designer Semi Circle Texture 3D Printed Hard Back Case Mo...","[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo'}]","[{'language_tag': 'en_IN', 'value': '3D Printed Hard Back Case Mobile Cover for Huawei P9 lite'}, {'language_tag': '...",[{'value': 'CELLULAR_PHONE_CASE'}],"[{'language_tag': 'en_IN', 'standardized_values': ['multi-colored'], 'value': 'Others'}]",NaN,NaN,71ZoXfUr-sL,"[81IiaAkE6UL, 61Xce1Hq7DL, 61ATVyzpLKL, 61oI69Yt4GL]"
5,B07LCHFZCW,"[{'language_tag': 'en_US', 'value': 'Amazon Kitchen, Cajun Style Potato Salad, 7.2 oz'}]","[{'language_tag': 'en_US', 'value': 'Amazon Go'}]",NaN,[{'value': 'GROCERY'}],NaN,NaN,NaN,61bCuBtuZ4L,"[61njweKAz7L, 61UEtH+yC3L]"
6,B077W2YX72,"[{'language_tag': 'en_US', 'value': 'Wickedly Prime Peanut Butter-Filled Pretzels, 44 Ounce'}]","[{'language_tag': 'en_US', 'value': 'Wickedly Prime'}]","[{'language_tag': 'en_US', 'value': 'One 44-ounce plastic tub'}, {'language_tag': 'en_US', 'value': 'Contains wheat,...",[{'value': 'GROCERY'}],NaN,NaN,NaN,71QbaO8qZIL,"[51M39MBAwDL, 71F6YzH37XL, 71d7-4rSVlL, 61ucPja6t0L]"
7,B07TH39LDF,"[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo Designer Cartoon Pattern 3D Printed Hard Back Case Mobile...","[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo'}]","[{'language_tag': 'en_IN', 'value': '3D Printed Hard Back Case Mobile Cover for Gionee S6s'}, {'language_tag': 'en_I...",[{'value': 'CELLULAR_PHONE_CASE'}],"[{'language_tag': 'en_IN', 'standardized_values': ['multi-colored'], 'value': 'Others'}]",NaN,NaN,710GWe3VMGL,"[61oI69Yt4GL, 61Xce1Hq7DL, 61ATVyzpLKL, 71Hr+ck1KuL]"
8,B07TG3WCBD,"[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo Designer Color Smoke 3D Printed Hard Back Case Mobile Cov...","[{'language_tag': 'en_IN', 'value': 'Amazon Brand - Solimo'}]","[{'language_tag': 'en_IN', 'value': '3D Printed Hard Back Case Mobile Cover for Micromax Canvas Sliver 5 Q450'}, {'l...",[{'value': 'CELLULAR_PHONE_CASE'}],"[{'language_tag': 'en_IN', 'standardized_values': ['multi-colored'], 'value': 'Others

## Corrected Image Mapping Validation Using images.csv.gz

The earlier direct mapping check compared listing image IDs directly with image file stems in `abo-images-small.tar`, which produced `mapped_image_count = 0`. The corrected path should validate `main_image_id` against `images/metadata/images.csv.gz`, then resolve the CSV `path` to an image tar member.

In [67]:
IMAGES_METADATA_MEMBER = "images/metadata/images.csv.gz"


def tar_member_exists(tar_path: Path, member_name: str) -> bool:
    """Check whether a tar member exists without extracting the archive."""
    if not tar_path.exists():
        return False

    with tarfile.open(tar_path, "r") as tar:
        try:
            tar.getmember(member_name)
            return True
        except KeyError:
            return False


images_metadata_exists = tar_member_exists(IMAGES_TAR_PATH, IMAGES_METADATA_MEMBER)
images_metadata_exists

True

In [68]:
def read_images_metadata_mapping(
    tar_path: Path,
    member_name: str = IMAGES_METADATA_MEMBER,
    required_columns: tuple[str, str] = ("image_id", "path"),
) -> pd.DataFrame:
    """Read only the image_id/path mapping columns from images.csv.gz inside the image tar."""
    if not tar_path.exists():
        return pd.DataFrame(columns=list(required_columns))

    with tarfile.open(tar_path, "r") as tar:
        try:
            member = tar.getmember(member_name)
        except KeyError:
            return pd.DataFrame(columns=list(required_columns))

        file_obj = tar.extractfile(member)
        if file_obj is None:
            return pd.DataFrame(columns=list(required_columns))

        with gzip.GzipFile(fileobj=file_obj) as gzip_file:
            metadata_df = pd.read_csv(gzip_file, usecols=list(required_columns), dtype=str)

    metadata_df = metadata_df.dropna(subset=["image_id", "path"])
    metadata_df["image_id"] = metadata_df["image_id"].str.strip()
    metadata_df["path"] = metadata_df["path"].str.strip()
    metadata_df = metadata_df.drop_duplicates(subset=["image_id"], keep="first")
    return metadata_df


images_metadata_mapping_df = read_images_metadata_mapping(IMAGES_TAR_PATH)
print(f"images.csv.gz mapping rows loaded: {len(images_metadata_mapping_df):,}")
images_metadata_mapping_df.head()

images.csv.gz mapping rows loaded: 398,212


,image_id,path
0,010-mllS7JL,14/14fe8812.jpg
1,01dkn0Gyx0L,da/daab0cad.jpg
2,01sUPg0387L,d2/d2daaae9.jpg
3,1168jc-5r1L,3a/3a4e88e6.jpg
4,11RUV5Fs65L,d9/d91ab9cf.jpg


In [69]:
def resolve_small_image_tar_path(csv_path: str | None) -> str | None:
    """Convert images.csv.gz path values to expected small-image tar member paths."""
    if csv_path is None or pd.isna(csv_path):
        return None

    normalized_path = str(csv_path).strip().lstrip("/")
    if not normalized_path:
        return None
    if normalized_path.startswith("images/small/"):
        return normalized_path
    return f"images/small/{normalized_path}"


image_id_to_csv_path = dict(
    zip(images_metadata_mapping_df["image_id"], images_metadata_mapping_df["path"])
)

sampled_main_image_ids = {
    str(record.get("main_image_id")).strip()
    for record in listing_records
    if record.get("main_image_id") is not None and str(record.get("main_image_id")).strip()
}

candidate_resolved_paths = {
    resolve_small_image_tar_path(image_id_to_csv_path[image_id])
    for image_id in sampled_main_image_ids
    if image_id in image_id_to_csv_path
}
candidate_resolved_paths.discard(None)

# Checking only the sampled candidate path set keeps this validation bounded and avoids extracting images.
resolved_paths_present_in_tar = set()
if IMAGES_TAR_PATH.exists() and candidate_resolved_paths:
    with tarfile.open(IMAGES_TAR_PATH, "r") as tar:
        for member in tar:
            if member.name in candidate_resolved_paths:
                resolved_paths_present_in_tar.add(member.name)
                if resolved_paths_present_in_tar == candidate_resolved_paths:
                    break

len(candidate_resolved_paths), len(resolved_paths_present_in_tar)

(491, 491)

In [70]:
corrected_mapping_rows = []
for record in listing_records:
    item_id = first_scalar(record.get("item_id"))
    main_image_id = record.get("main_image_id")
    main_image_id = str(main_image_id).strip() if main_image_id is not None and str(main_image_id).strip() else None

    csv_path = image_id_to_csv_path.get(main_image_id) if main_image_id else None
    resolved_tar_path = resolve_small_image_tar_path(csv_path)
    path_exists_in_tar = resolved_tar_path in resolved_paths_present_in_tar if resolved_tar_path else False
    usable_text = bool(combined_text(record))

    if main_image_id is None:
        failure_reason = "missing_main_image_id"
    elif csv_path is None:
        failure_reason = "image_id_not_found_in_csv"
    elif not path_exists_in_tar:
        failure_reason = "path_not_found_in_tar"
    else:
        failure_reason = None

    corrected_mapping_rows.append(
        {
            "item_id": item_id,
            "main_image_id": main_image_id,
            "main_image_id_available": main_image_id is not None,
            "main_image_id_found_in_images_csv": csv_path is not None,
            "csv_path": csv_path,
            "resolved_tar_path": resolved_tar_path,
            "image_path_exists_in_tar": path_exists_in_tar,
            "has_usable_text": usable_text,
            "corrected_clip_ready": usable_text and path_exists_in_tar,
            "failure_reason": failure_reason,
        }
    )

corrected_mapping_df = pd.DataFrame(corrected_mapping_rows)
corrected_mapping_df.head()

,item_id,main_image_id,main_image_id_available,main_image_id_found_in_images_csv,csv_path,resolved_tar_path,image_path_exists_in_tar,has_usable_text,corrected_clip_ready,failure_reason
0,B07NQ437BB,61fH9aTfMIL,True,True,05/051b1105.jpg,images/small/05/051b1105.jpg,True,True,True,NaN
1,B0857LSVB7,81-DuD5XzmL,True,True,39/39df5b7d.jpg,images/small/39/39df5b7d.jpg,True,True,True,NaN
2,B07C5FF8QS,81YCp3dcurL,True,True,0f/0f039d0e.jpg,images/small/0f/0f039d0e.jpg,True,True,True,NaN
3,B07K591232,71EwvyC4A1L,True,True,ea/ea42fa6f.jpg,images/small/ea/ea42fa6f.jpg,True,True,True,NaN
4,B07TG425LX,71ZoXfUr-sL,True,True,56/562f90f1.jpg,images/small/56/562f90f1.jpg,True,True,True,NaN


In [71]:
corrected_image_mapping_summary_df = pd.DataFrame(
    [
        {
            "sampled_records": int(len(corrected_mapping_df)),
            "main_image_id_available_count": int(corrected_mapping_df["main_image_id_available"].sum()),
            "main_image_id_found_in_images_csv_count": int(corrected_mapping_df["main_image_id_found_in_images_csv"].sum()),
            "image_path_exists_in_tar_count": int(corrected_mapping_df["image_path_exists_in_tar"].sum()),
            "corrected_mapped_image_count": int(corrected_mapping_df["image_path_exists_in_tar"].sum()),
            "corrected_clip_ready_count": int(corrected_mapping_df["corrected_clip_ready"].sum()),
        }
    ]
)
corrected_image_mapping_summary_df

,sampled_records,main_image_id_available_count,main_image_id_found_in_images_csv_count,image_path_exists_in_tar_count,corrected_mapped_image_count,corrected_clip_ready_count
0,500,495,495,495,495,495


In [72]:
successful_mapping_examples = corrected_mapping_df.loc[
    corrected_mapping_df["image_path_exists_in_tar"],
    ["item_id", "main_image_id", "csv_path", "resolved_tar_path"],
].head(5)

successful_mapping_examples

,item_id,main_image_id,csv_path,resolved_tar_path
0,B07NQ437BB,61fH9aTfMIL,05/051b1105.jpg,images/small/05/051b1105.jpg
1,B0857LSVB7,81-DuD5XzmL,39/39df5b7d.jpg,images/small/39/39df5b7d.jpg
2,B07C5FF8QS,81YCp3dcurL,0f/0f039d0e.jpg,images/small/0f/0f039d0e.jpg
3,B07K591232,71EwvyC4A1L,ea/ea42fa6f.jpg,images/small/ea/ea42fa6f.jpg
4,B07TG425LX,71ZoXfUr-sL,56/562f90f1.jpg,images/small/56/562f90f1.jpg


In [73]:
failed_mapping_examples = corrected_mapping_df.loc[
    ~corrected_mapping_df["image_path_exists_in_tar"],
    ["item_id", "main_image_id", "csv_path", "resolved_tar_path", "failure_reason"],
].head(5)

failed_mapping_examples

,item_id,main_image_id,csv_path,resolved_tar_path,failure_reason
9,B07TL47SLR,NaN,NaN,NaN,missing_main_image_id
266,B084MSLSB1,NaN,NaN,NaN,missing_main_image_id
401,B07LH66X3V,NaN,NaN,NaN,missing_main_image_id
417,B08BHX9NCY,NaN,NaN,NaN,missing_main_image_id
456,B08DNW9BY9,NaN,NaN,NaN,missing_main_image_id


Image mapping feasibility note: if `corrected_mapped_image_count` is greater than zero, ABO image mapping is feasible through `images/metadata/images.csv.gz` using `main_image_id -> image_id -> path -> images/small/{path}`. The next step should be to document the approved mapping rule before creating any cleaning script.

## 17. Draft Cleaning Rule Decisions

These are draft decisions to validate after inspecting the real sampled outputs above. They are not implemented as a cleaning script in this task.

- Preserve `item_id` as the primary ABO product identifier if it is present and unique enough in the sampled records.
- Build product text from available listing fields such as `item_name`, `brand`, `bullet_point`, `product_type`, `color`, `material`, and `style`.
- Treat empty strings, empty lists, and missing values as missing text components.
- Keep `main_image_id` as the preferred image field when available.
- Use additional image ID fields only when they map to actual image files in the small image archive.
- Exclude records from future CLIP-style evaluation if they lack either usable product text or a mapped image.
- Do not infer missing brands, product types, image IDs, or cross-dataset identifiers.
- Keep the future cleaned product table ABO-only and provenance-aware.

## 18. Final Summary and Next Step

In [74]:
inspection_summary = {
    "listings_tar_exists": LISTINGS_TAR_PATH.exists(),
    "images_tar_exists": IMAGES_TAR_PATH.exists(),
    "sample_abo_dir_exists": SAMPLE_ABO_DIR.exists(),
    "sampled_listing_records": int(len(listings_df)),
    "observed_listing_fields": sorted(listings_df.columns.tolist()),
    "image_archive_file_count": int(len(available_image_ids)),
    "non_null_item_ids": int(duplicate_summary["non_null_item_ids"]),
    "unique_item_ids": int(duplicate_summary["unique_item_ids"]),
    "duplicate_item_id_rows": duplicate_summary["duplicate_item_id_rows"],
    "usable_text_count": int(clip_ready_df["has_usable_text"].sum()) if not clip_ready_df.empty else 0,
    "mapped_image_count": int(clip_ready_df["has_mapped_image"].sum()) if not clip_ready_df.empty else 0,
    "clip_ready_count": int(clip_ready_df["is_clip_ready"].sum()) if not clip_ready_df.empty else 0,
}

inspection_summary

{'listings_tar_exists': True,
 'images_tar_exists': True,
 'sample_abo_dir_exists': True,
 'sampled_listing_records': 500,
 'observed_listing_fields': ['3dmodel_id',
  'brand',
  'bullet_point',
  'color',
  'color_code',
  'country',
  'domain_name',
  'fabric_type',
  'finish_type',
  'item_dimensions',
  'item_id',
  'item_keywords',
  'item_name',
  'item_shape',
  'item_weight',
  'main_image_id',
  'marketplace',
  'material',
  'model_name',
  'model_number',
  'model_year',
  'node',
  'other_image_id',
  'pattern',
  'product_description',
  'product_type',
  'spin_id',
  'style'],
 'image_archive_file_count': 398215,
 'non_null_item_ids': 500,
 'unique_item_ids': 500,
 'duplicate_item_id_rows': 0,
 'usable_text_count': 500,
 'mapped_image_count': 0,
 'clip_ready_count': 0}

Next step after review: convert approved inspection findings into a small, production-oriented ABO data-cleaning design before writing any cleaning script.